# LeetCode #1458: Max Dot Product of Two Subsequences

https://leetcode.com/problems/max-dot-product-of-two-subsequences/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^{m+n})$ | $O(m+n)$ |
| **Optimal: 2-D DP ★** | $O(m \times n)$ | $O(m \times n)$ |

---

## Understanding the Methods

### Brute Force
Enumerate all subsequences of both arrays and compute dot products. Exponential time.

### Optimal: 2-D DP ★
`dp[i][j]` = maximum dot product using any non-empty subsequence of `nums1[0..i]` and any non-empty subsequence of `nums2[0..j]`. Transition: either pair element `i` with element `j` (possibly extending a prior pairing) or skip one element from either array.

**Constraints:**
* $1 \leq nums1.length, nums2.length \leq 500$
* $-1000 \leq nums1[i], nums2[j] \leq 1000$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxDotProduct(int[] nums1, int[] nums2) {
        int m = nums1.Length, n = nums2.Length;
        int[,] dp = new int[m + 1, n + 1];

        // Initialize to negative infinity (no pair chosen yet)
        for (int i = 0; i <= m; i++)
            for (int j = 0; j <= n; j++)
                dp[i, j] = int.MinValue / 2;

        for (int i = 1; i <= m; i++) {
            for (int j = 1; j <= n; j++) {
                // Pair nums1[i-1] with nums2[j-1], optionally extending a prior pairing
                int pair = nums1[i - 1] * nums2[j - 1];
                dp[i, j] = Math.Max(pair,
                            Math.Max(pair + Math.Max(0, dp[i - 1, j - 1]),
                            Math.Max(dp[i - 1, j], dp[i, j - 1])));
            }
        }
        return dp[m, n];
    }
}

### Python

In [ ]:
class Solution:
    def max_dot_product(self, nums1: list[int], nums2: list[int]) -> int:
        m, n = len(nums1), len(nums2)
        NEG_INF = float('-inf')
        dp = [[NEG_INF] * (n + 1) for _ in range(m + 1)]

        for i in range(1, m + 1):
            for j in range(1, n + 1):
                # Pair nums1[i-1] with nums2[j-1], optionally extending a prior pairing
                pair = nums1[i - 1] * nums2[j - 1]
                prev = dp[i - 1][j - 1] if dp[i - 1][j - 1] != NEG_INF else 0
                dp[i][j] = max(
                    pair + max(0, prev),  # add prior best if positive, else start fresh
                    dp[i - 1][j],         # skip nums1[i-1]
                    dp[i][j - 1],         # skip nums2[j-1]
                )

        return dp[m][n]

### Go

In [ ]:
func maxDotProduct(nums1 []int, nums2 []int) int {
    m, n := len(nums1), len(nums2)
    const NEG_INF = -1 << 30
    dp := make([][]int, m+1)
    for i := range dp {
        dp[i] = make([]int, n+1)
        for j := range dp[i] { dp[i][j] = NEG_INF }
    }

    for i := 1; i <= m; i++ {
        for j := 1; j <= n; j++ {
            pair := nums1[i-1] * nums2[j-1]
            prev := 0
            if dp[i-1][j-1] > 0 { prev = dp[i-1][j-1] }
            // Pair nums1[i-1] with nums2[j-1], optionally extending a prior pairing
            best := pair + prev
            if dp[i-1][j] > best { best = dp[i-1][j] }
            if dp[i][j-1] > best { best = dp[i][j-1] }
            dp[i][j] = best
        }
    }
    return dp[m][n]
}

### Rust

In [ ]:
impl Solution {
    pub fn max_dot_product(nums1: Vec<i32>, nums2: Vec<i32>) -> i32 {
        let (m, n) = (nums1.len(), nums2.len());
        const NEG_INF: i32 = i32::MIN / 2;
        let mut dp = vec![vec![NEG_INF; n + 1]; m + 1];

        for i in 1..=m {
            for j in 1..=n {
                let pair = nums1[i - 1] * nums2[j - 1];
                // Extend prior best if positive, otherwise start a new pairing
                let prev = if dp[i-1][j-1] > 0 { dp[i-1][j-1] } else { 0 };
                dp[i][j] = (pair + prev)
                    .max(dp[i - 1][j])
                    .max(dp[i][j - 1]);
            }
        }
        dp[m][n]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums1 = [2,1,-2,5]`, `nums2 = [3,0,-6]`
Pair $5 \times (-6) = -30$ is bad; pair $(-2) \times (-6) = 12$ and $5 \times 3 = 15$ can both be used. Best: $(-2) \times (-6) + 5 \times 3 = 12 + 15 = 27$? No — subsequences must keep order. Pair $2 \times 3 = 6$ then $5 \times (-6)$? No. Best is $\{(-2, -6), (5, ?)\}$... the answer is $18$ ($(-2) \times (-6) = 12$ is the single best pair giving 18 when combined with $2 \times 3$).

### 2. Slightly Complex
**Input:** `nums1 = [3,-2]`, `nums2 = [2,-6,7]`
Options: $3 \times 7 = 21$, $3 \times (-6) + (-2) \times 7 = -32$, best single pair $3 \times 7 = 21$. Answer: $21$.

### 3. Edge Case: Time Factor
**Input:** `nums1` and `nums2` each have 500 elements
The DP fills a $500 \times 500 = 250{,}000$ cell table in a single double loop — $O(m \times n)$ with no recursion overhead.

### 4. Edge Case: Space Factor
**Input:** Same 500×500 input
The table requires $250{,}000$ integers. Space can be reduced to $O(n)$ using rolling arrays, but the base solution uses $O(m \times n)$.

### 5. Almost-Impossible but Plausible
**Input:** All elements in both arrays are $-1000$
Every pair product is $10^6 > 0$, so the DP eagerly accumulates pairs. The maximum is achieved by pairing all $\min(m,n)$ elements: total $= \min(m,n) \times 10^6$. With $m = n = 500$, the answer is $5 \times 10^8$ — within int range.